# 03 — Model Training

Trains the CreditBridge XGBoost classifier with:
- **Monotone constraints** to enforce domain logic (more payments = better)
- **SMOTE** oversampling to address class imbalance
- **Platt scaling** calibration for reliable probability estimates
- **MLflow** experiment tracking

Target metrics: AUC ≥ 0.88, KS statistic ≥ 0.40

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV, FrozenEstimator, calibration_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

from src.model.train import preprocess_features
from src.model.predict import predict_score_details
from src.model.evaluate import evaluate_model

PROCESSED_PATH = '../data/processed/features.parquet'
df = pd.read_parquet(PROCESSED_PATH)
print(f'Loaded feature matrix: {df.shape}')

## 1. Train / Val / Test Split (70 / 15 / 15)

In [ ]:
FEATURE_COLS = [
    'gender_M', 'geography_urban', 'geography_semi_urban', 'geography_rural',
    'income_high', 'income_mid', 'income_low', 'is_msme_int',
    'upi_txn_count_6m', 'upi_consistency_score', 'upi_merchant_diversity',
    'upi_failed_rate', 'upi_avg_txn_value', 'upi_night_txn_share', 'upi_income_regularity',
    'utility_streak_length', 'utility_days_before_due_avg',
    'utility_lapse_count_12m', 'utility_reinstatement_count',
    'mobile_plan_tier', 'mobile_recharge_streak', 'mobile_plan_trend', 'mobile_lapse_count',
    'gst_filing_regularity', 'gst_turnover_trend', 'gst_penalty_count',
]

df_prep = preprocess_features(df)
available_cols = [c for c in FEATURE_COLS if c in df_prep.columns]
X = df_prep[available_cols]
y = df_prep['default_label']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1765, stratify=y_train_full, random_state=42
)

print(f'Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}')
print(f'Train default rate: {y_train.mean():.2%}')

## 2. SMOTE Oversampling

In [ ]:
smote = SMOTE(k_neighbors=5, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {X_train.shape[0]:,} samples, default rate: {y_train.mean():.2%}')
print(f'After  SMOTE: {X_train_res.shape[0]:,} samples, default rate: {y_train_res.mean():.2%}')

## 3. XGBoost with Monotone Constraints

In [ ]:
MONOTONE = {
    'upi_consistency_score': 1,    # more = better
    'utility_streak_length': 1,
    'mobile_recharge_streak': 1,
    'upi_income_regularity': 1,
    'gst_filing_regularity': 1,
    'upi_failed_rate': -1,         # more = worse
    'utility_lapse_count_12m': -1,
    'mobile_lapse_count': -1,
    'gst_penalty_count': -1,
    'utility_reinstatement_count': -1,
}
constraints = tuple(MONOTONE.get(c, 0) for c in available_cols)

base_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    monotone_constraints=constraints,
    scale_pos_weight=3,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    verbosity=0,
)

print('Fitting XGBoost...')
base_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print('Done.')

## 4. Platt Scaling Calibration

In [ ]:
calibrated_model = CalibratedClassifierCV(
    estimator=FrozenEstimator(base_model), method='sigmoid'
)
calibrated_model.fit(X_val, y_val)
print('Platt calibration fitted on validation set.')

## 5. Test Set Evaluation

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

y_prob = calibrated_model.predict_proba(X_test)[:, 1]
metrics = evaluate_model(y_test.values, y_prob, output_dir='../models')

print(f'\nAUC  : {metrics["auc"]:.4f}  (target >= 0.88)')
print(f'KS   : {metrics["ks"]:.4f}  (target >= 0.40)')
print(f'ECE  : {metrics["ece"]:.4f}')
print(f'Brier: {metrics["brier_score"]:.4f}')

## 6. Reliability Curve

In [ ]:
prob_true, prob_pred = calibration_curve(y_test.values, y_prob, n_bins=10)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
ax.plot(prob_pred, prob_true, 'o-', color='#0066FF', linewidth=2, markersize=7, label='Calibrated model')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of defaults')
ax.set_title('Reliability Curve (Platt Calibration)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Score Banding Distribution

In [ ]:
from src.model.predict import probability_to_score, score_to_band

scores = [probability_to_score(p) for p in y_prob]
bands = [score_to_band(s) for s in scores]

band_order = ['Prime', 'Near-prime', 'Subprime', 'High risk', 'Decline']
band_colors = {'Prime': '#00C853', 'Near-prime': '#69F0AE', 'Subprime': '#FFD740',
               'High risk': '#FF6D00', 'Decline': '#D50000'}

import collections
band_counts = collections.Counter(bands)

fig, ax = plt.subplots(figsize=(8, 4))
vals = [band_counts.get(b, 0) for b in band_order]
colors = [band_colors[b] for b in band_order]
bars = ax.bar(band_order, vals, color=colors, edgecolor='#0A0A0A', linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Score Band Distribution (Test Set)', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 8. Feature Importances (XGBoost Gain)

In [ ]:
import_df = pd.Series(
    base_model.feature_importances_, index=available_cols
).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(import_df.index, import_df.values, color='#0066FF', edgecolor='#0A0A0A', linewidth=0.8)
ax.set_title('Top-15 XGBoost Feature Importances (Gain)', fontweight='bold')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.savefig('../models/feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()